# Deliverable 7: Certified Test-Set Robustness via MinSwitch

### Selected Models for Deliverable 7
Following the empirical survey of Deliverable 6, we evaluate the **3 fully completed models** ($100\%$ full forest coverage, `Completed == True`):

| Dataset | Trees ($D$) | Features ($F$) | Classes ($C$) | Optimal Budget | Optimal Selector | Full Completion | Test Split Samples |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **`threeOf9`** | 44 | 9 | 2 | **100** | **`info_gain`** | **True (44/44)** | 103 |
| **`lupus`** | 28 | 3 | 2 | **100** | **`random_node`** | **True (28/28)** | 18 |
| **`mammo`** | 12 | 13 | 2 | **50** | **`info_gain`** | **True (12/12)** | 193 |

In [13]:
import os
import sys
import time
import random
import joblib
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

# Ensure project root is in sys.path
curr = os.path.abspath(os.getcwd())
while curr and curr != os.path.dirname(curr):
    if os.path.isdir(os.path.join(curr, "rfbdd")) and os.path.isdir(os.path.join(curr, "datasets")):
        project_root = curr
        break
    curr = os.path.dirname(curr)
else:
    project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from rfbdd import (
    load_forest,
    load_dataset,
    discretize,
    run_pipeline,
)
from rfbdd.robustness import compute_sample_robustness

# Create results directory for Deliverable 7
results_del7_dir = os.path.join(project_root, "results", "deliverable_7")
os.makedirs(results_del7_dir, exist_ok=True)
print(f"Deliverable 7 results directory: {results_del7_dir}")

# Selected models specification
MODELS_CONFIG = [
    {"dataset": "threeOf9", "budget": 100, "selector": "info_gain", "timeout": 15.0, "seed": 42},
    {"dataset": "lupus",    "budget": 100, "selector": "random_node", "timeout": 15.0, "seed": 2},
    {"dataset": "mammo",    "budget": 50,  "selector": "info_gain", "timeout": 15.0, "seed": 42},
]
print("Configured target models:", [m["dataset"] for m in MODELS_CONFIG])

Deliverable 7 results directory: c:\Users\rikyf\OneDrive\Desktop\rf-obdd-robustness\results\deliverable_7
Configured target models: ['threeOf9', 'lupus', 'mammo']


## 1. Test-Set Robustness Evaluation Engine

For each of the three models, the engine:
1. **Compiles the Complete Forest** into its budgeted frontier leaves under the optimal selector and budget.
2. **Evaluates MinSwitch (Theorem 2 & Proposition 11)** on every sample $\hat{x}$ of the held-out test split `<name>_test.csv`:
   - **Targeted radius** $\text{rob}_\ell(\hat{x})$ for each rival label $\ell$.
   - **Untargeted radius** $\min_{\ell \neq \ell^*} \text{rob}_\ell(\hat{x})$ to the nearest decision boundary.
   - **Certified class**: The label certified at $\text{rob}(\hat{x}) = 0$.
   - **Minimal bit-switches**: The specific input cell variables flipped on the shortest path.
   - **Touched continuous features**: Mapping minimal bit flips back to continuous feature axes.
3. **Saves sample-level results** to dedicated CSV files in `results/deliverable_7/`.

In [ ]:
def evaluate_model_test_robustness(dataset_name: str, budget: int, selector: str, timeout: float = 15.0, seed: int = 42):
    """Run full forest compilation and MinSwitch evaluation over the held-out test split."""
    print("=" * 80)
    print(f" EVALUATING TEST-SET ROBUSTNESS: {dataset_name.upper()}")
    print(f" Configuration: Budget B={budget} | Selector='{selector}' | Timeout={timeout}s")
    print("=" * 80)
    
    random.seed(seed)
    np.random.seed(seed)
    
    # Load Forest and Discretization
    pkl_path = os.path.join(project_root, "datasets", dataset_name, f"{dataset_name}_rf.pkl")
    forest = load_forest(pkl_path)
    disc = discretize(forest)
    
    # Check label alignment with trained RF model
    pkl_dict = joblib.load(pkl_path)
    rf_model = pkl_dict.get("model", None)
    expected_acc = pkl_dict.get("test_acc", None)
    
    # Compile Forest into Budgeted Frontier
    t_start_compile = time.perf_counter()
    pipeline_res = run_pipeline(
        dataset=forest,
        budget=budget,
        selector=selector,
        timeout_seconds=timeout,
        max_leaves=500,
        compute_readouts=False,
        compute_rankings=False,
        evaluate_robustness=False,
        verbose=False,
    )
    compile_time = time.perf_counter() - t_start_compile
    st = pipeline_res.stats
    leaves = pipeline_res.leaves
    bdd = pipeline_res.bdd
    
    print(f"[*] Compilation complete in {compile_time:.2f}s | Completed: {st.completed} | Leaves: {len(leaves)} | Peak Acc Size: {st.peak_acc_size}")
    
    # Load Held-Out Test Split
    test_path = os.path.join(project_root, "datasets", dataset_name, f"{dataset_name}_test.csv")
    df_test = pd.read_csv(test_path)
    X_test = df_test.iloc[:, :-1].values
    y_test = df_test.iloc[:, -1].values
    n_samples = len(X_test)
    
    # Detect label orientation: if RF predicts inverted binary labels compared to CSV, align them
    label_flip_needed = False
    if rf_model is not None and forest.n_classes == 2:
        rf_preds = rf_model.predict(X_test)
        direct_acc = float((rf_preds == y_test).mean())
        inverted_acc = float((rf_preds == (1 - y_test)).mean())
        if inverted_acc > direct_acc:
            label_flip_needed = True
            print(f"[*] Label alignment detected inverted binary target in test CSV (direct: {direct_acc:.1%}, inverted: {inverted_acc:.1%}). Aligning true labels with model classes.")
            y_test = 1 - y_test
    
    print(f"[*] Loaded test split: {n_samples} samples across {forest.n_features} features.")
    
    # Evaluate MinSwitch on every test sample
    t_start_rob = time.perf_counter()
    sample_records = []
    feature_flip_counts = {j: 0 for j in range(forest.n_features)}
    
    for idx in range(n_samples):
        sample = X_test[idx]
        true_label = int(y_test[idx])
        
        rob = compute_sample_robustness(
            bdd=bdd,
            representation=leaves,
            sample=sample,
            disc=disc,
            n_classes=forest.n_classes,
        )
        
        cert_label = rob["certified_class"]
        untargeted_r = rob["untargeted_radius"]
        nearest_rival = rob["nearest_rival"]
        
        # Track minimal flips for nearest rival
        flips_nearest = rob["flips_per_label"].get(nearest_rival, []) if nearest_rival is not None else []
        features_nearest = rob["features_per_label"].get(nearest_rival, []) if nearest_rival is not None else []
        
        for feat_idx in features_nearest:
            feature_flip_counts[feat_idx] = feature_flip_counts.get(feat_idx, 0) + 1
            
        record = {
            "Sample_Index": idx,
            "Dataset": dataset_name,
            "True_Label": true_label,
            "Certified_Label": cert_label,
            "Correctly_Certified": bool(true_label == cert_label),
            "Untargeted_Radius": untargeted_r,
            "Nearest_Rival_Label": nearest_rival,
            "Bit_Flips_Count": len(flips_nearest),
            "Flipped_Bits": ";".join(flips_nearest),
            "Touched_Features": ";".join(map(str, features_nearest)),
            "Num_Touched_Features": len(features_nearest),
            "Is_Fragile": bool(0 < untargeted_r <= 1.0),
        }
        
        # Add targeted radii for all classes
        for c_idx in range(forest.n_classes):
            record[f"Targeted_Radius_Class_{c_idx}"] = rob["targeted_radii"].get(c_idx, np.nan)
            
        sample_records.append(record)
        
    total_rob_time = time.perf_counter() - t_start_rob
    avg_time_per_sample_ms = (total_rob_time / n_samples) * 1000
    
    df_samples = pd.DataFrame(sample_records)
    
    # Save sample-level results to CSV
    csv_out = os.path.join(results_del7_dir, f"{dataset_name}_test_robustness.csv")
    df_samples.to_csv(csv_out, index=False)
    print(f"[+] Test robustness evaluation completed in {total_rob_time:.3f}s ({avg_time_per_sample_ms:.2f} ms/sample)!")
    print(f"    Saved sample-level results -> {csv_out}")
    
    return df_samples, feature_flip_counts, forest, disc

## 2. Test-Set Robustness Evaluation: `threeOf9`

- **Dataset**: `threeOf9` (Synthetic parity benchmark, $D=44$ trees, $F=9$ features, $C=2$ classes)
- **Optimal Configuration**: Budget $B=100$, Selector `info_gain`
- **Full Forest Coverage**: $100\%$ ($k=44/44$ trees completed)

In [15]:
df_threeOf9, flips_threeOf9, forest_threeOf9, disc_threeOf9 = evaluate_model_test_robustness(
    dataset_name="threeOf9",
    budget=100,
    selector="info_gain",
    timeout=15.0,
    seed=42,
)
df_threeOf9.head(10)

 EVALUATING TEST-SET ROBUSTNESS: THREEOF9
 Configuration: Budget B=100 | Selector='info_gain' | Timeout=15.0s


c:\Users\rikyf\OneDrive\Desktop\rf-obdd-robustness\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.4.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\rikyf\OneDrive\Desktop\rf-obdd-robustness\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.4.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


[*] Compilation complete in 2.12s | Completed: True | Leaves: 7 | Peak Acc Size: 130
[*] Label alignment detected inverted binary target in test CSV (direct: 0.0%, inverted: 100.0%). Aligning true labels with model classes.
[*] Loaded test split: 103 samples across 9 features.
[+] Test robustness evaluation completed in 0.018s (0.17 ms/sample)!
    Saved sample-level results -> c:\Users\rikyf\OneDrive\Desktop\rf-obdd-robustness\results\deliverable_7\threeOf9_test_robustness.csv


,Sample_Index,Dataset,True_Label,Certified_Label,Correctly_Certified,Untargeted_Radius,Nearest_Rival_Label,Bit_Flips_Count,Flipped_Bits,Touched_Features,Num_Touched_Features,Is_Fragile,Targeted_Radius_Class_0,Targeted_Radius_Class_1
0,0,threeOf9,1,1,True,1.0,0,1,x0_0,0,1,True,1.0,0.0
1,1,threeOf9,0,0,True,1.0,1,1,x0_0,0,1,True,0.0,1.0
2,2,threeOf9,1,1,True,1.0,0,1,x0_0,0,1,True,1.0,0.0
3,3,threeOf9,0,0,True,1.0,1,1,x0_0,0,1,True,0.0,1.0
4,4,threeOf9,1,1,True,1.0,0,1,x0_0,0,1,True,1.0,0.0
5,5,threeOf9,0,0,True,1.0,1,1,x0_0,0,1,True,0.0,1.0
6,6,threeOf9,0,0,True,1.0,1,1,x0_0,0,1,True,0.0,1.0
7,7,threeOf9,0,0,True,1.0,1,1,x0_0,0,1,True,0.0,1.0
8,8,threeOf9,0,0,True,1.0,1,1,x0_0,0,1,True,0.0,1.0
9,9,threeOf9,0,0,True,1.0,1,1,x0_0,0,1,True,0.0,1.0


## 3. Test-Set Robustness Evaluation: `lupus`

- **Dataset**: `lupus` (Clinical immunology prognosis, $D=28$ trees, $F=3$ features, $C=2$ classes)
- **Optimal Configuration**: Budget $B=100$, Selector `random_node`
- **Full Forest Coverage**: $100\%$ ($k=28/28$ trees completed)

In [16]:
df_lupus, flips_lupus, forest_lupus, disc_lupus = evaluate_model_test_robustness(
    dataset_name="lupus",
    budget=100,
    selector="random_node",
    timeout=15.0,
    seed=2,
)
df_lupus.head(10)

 EVALUATING TEST-SET ROBUSTNESS: LUPUS
 Configuration: Budget B=100 | Selector='random_node' | Timeout=15.0s


c:\Users\rikyf\OneDrive\Desktop\rf-obdd-robustness\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.4.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\rikyf\OneDrive\Desktop\rf-obdd-robustness\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.4.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


[*] Compilation complete in 4.50s | Completed: True | Leaves: 92 | Peak Acc Size: 120
[*] Loaded test split: 18 samples across 3 features.
[+] Test robustness evaluation completed in 0.021s (1.14 ms/sample)!
    Saved sample-level results -> c:\Users\rikyf\OneDrive\Desktop\rf-obdd-robustness\results\deliverable_7\lupus_test_robustness.csv


,Sample_Index,Dataset,True_Label,Certified_Label,Correctly_Certified,Untargeted_Radius,Nearest_Rival_Label,Bit_Flips_Count,Flipped_Bits,Touched_Features,Num_Touched_Features,Is_Fragile,Targeted_Radius_Class_0,Targeted_Radius_Class_1
0,0,lupus,1,0,False,1.0,1,1,x1_3,1,1,True,0.0,1.0
1,1,lupus,1,1,True,1.0,0,1,x0_3,0,1,True,1.0,0.0
2,2,lupus,0,1,False,1.0,0,1,x0_4,0,1,True,1.0,0.0
3,3,lupus,1,0,False,1.0,1,1,x1_3,1,1,True,0.0,1.0
4,4,lupus,0,0,True,2.0,1,2,x1_2;x0_4,0;1,2,False,0.0,2.0
5,5,lupus,0,0,True,2.0,1,2,x1_3;x0_4,0;1,2,False,0.0,2.0
6,6,lupus,0,0,True,2.0,1,2,x1_3;x0_1,0;1,2,False,0.0,2.0
7,7,lupus,0,0,True,3.0,1,3,x0_2;x0_1;x0_4,0,1,False,0.0,3.0
8,8,lupus,0,0,True,4.0,1,4,x1_2;x2_1;x0_2;x1_3,0;1;2,3,False,0.0,4.0
9,9,lupus,0,0,True,2.0,1,2,x2_3;x1_3,1;2,2,False,0.0,2.0


## 4. Test-Set Robustness Evaluation: `mammo`

- **Dataset**: `mammo` (Mammographic mass malignancy detection, $D=12$ trees, $F=13$ features, $C=2$ classes)
- **Optimal Configuration**: Budget $B=50$, Selector `info_gain`
- **Full Forest Coverage**: $100\%$ ($k=12/12$ trees completed)

In [17]:
df_mammo, flips_mammo, forest_mammo, disc_mammo = evaluate_model_test_robustness(
    dataset_name="mammo",
    budget=50,
    selector="info_gain",
    timeout=15.0,
    seed=42,
)
df_mammo.head(10)

 EVALUATING TEST-SET ROBUSTNESS: MAMMO
 Configuration: Budget B=50 | Selector='info_gain' | Timeout=15.0s


c:\Users\rikyf\OneDrive\Desktop\rf-obdd-robustness\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.4.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\rikyf\OneDrive\Desktop\rf-obdd-robustness\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.4.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


[*] Compilation complete in 4.99s | Completed: True | Leaves: 119 | Peak Acc Size: 83
[*] Label alignment detected inverted binary target in test CSV (direct: 21.2%, inverted: 78.8%). Aligning true labels with model classes.
[*] Loaded test split: 193 samples across 13 features.
[+] Test robustness evaluation completed in 0.150s (0.78 ms/sample)!
    Saved sample-level results -> c:\Users\rikyf\OneDrive\Desktop\rf-obdd-robustness\results\deliverable_7\mammo_test_robustness.csv


,Sample_Index,Dataset,True_Label,Certified_Label,Correctly_Certified,Untargeted_Radius,Nearest_Rival_Label,Bit_Flips_Count,Flipped_Bits,Touched_Features,Num_Touched_Features,Is_Fragile,Targeted_Radius_Class_0,Targeted_Radius_Class_1
0,0,mammo,1,1,True,2.0,0,2,x4_0;x8_0,4;8,2,False,2.0,0.0
1,1,mammo,1,1,True,1.0,0,1,x1_0,1,1,True,1.0,0.0
2,2,mammo,0,0,True,1.0,1,1,x3_0,3,1,True,0.0,1.0
3,3,mammo,0,0,True,1.0,1,1,x10_0,10,1,True,0.0,1.0
4,4,mammo,1,1,True,2.0,0,2,x4_0;x0_0,0;4,2,False,2.0,0.0
5,5,mammo,1,1,True,2.0,0,2,x4_0;x8_0,4;8,2,False,2.0,0.0
6,6,mammo,0,0,True,1.0,1,1,x3_0,3,1,True,0.0,1.0
7,7,mammo,1,1,True,1.0,0,1,x11_0,11,1,True,1.0,0.0
8,8,mammo,1,1,True,2.0,0,2,x4_0;x0_0,0;4,2,False,2.0,0.0
9,9,mammo,1,1,True,1.0,0,1,x4_0,4,1,True,1.0,0.0


## 5. Comparative Analysis & Robustness Distributions

Aggregating over the test sets answers the fundamental Deliverable 7 questions:
1. **Certified Test Accuracy**: Fraction of test samples where the certified class equals the ground-truth label.
2. **Radius Distribution**: Mean, median, and spread of untargeted bit-switch distances.
3. **Fragility Analysis**: Number and percentage of highly fragile test points (samples sitting at distance $\text{rob} = 1$ from the rival boundary).
4. **Variable Vulnerability**: Identifying the features most frequently perturbed in minimal adversarial switches.

In [18]:
# Aggregate Metrics across the 3 models
summary_rows = []

for df_curr, ds_name in [(df_threeOf9, "threeOf9"), (df_lupus, "lupus"), (df_mammo, "mammo")]:
    n_total = len(df_curr)
    cert_acc = df_curr["Correctly_Certified"].mean() * 100
    mean_r = df_curr["Untargeted_Radius"].mean()
    median_r = df_curr["Untargeted_Radius"].median()
    min_r = df_curr["Untargeted_Radius"].min()
    max_r = df_curr["Untargeted_Radius"].max()
    fragile_count = df_curr["Is_Fragile"].sum()
    fragile_pct = (fragile_count / n_total) * 100
    avg_touched_feats = df_curr["Num_Touched_Features"].mean()
    
    summary_rows.append({
        "Dataset": ds_name,
        "Test_Samples": n_total,
        "Certified_Accuracy_Pct": round(cert_acc, 2),
        "Mean_Untargeted_Radius": round(mean_r, 2),
        "Median_Untargeted_Radius": round(median_r, 2),
        "Min_Radius": int(min_r),
        "Max_Radius": int(max_r),
        "Fragile_Samples_Count": int(fragile_count),
        "Fragile_Samples_Pct": round(fragile_pct, 1),
        "Avg_Touched_Features": round(avg_touched_feats, 2),
    })

df_summary_del7 = pd.DataFrame(summary_rows)
summary_csv = os.path.join(results_del7_dir, "deliverable_7_summary_metrics.csv")
df_summary_del7.to_csv(summary_csv, index=False)

print("=" * 95)
print(" DELIVERABLE 7: CERTIFIED TEST-SET ROBUSTNESS SUMMARY")
print("=" * 95)
print(df_summary_del7.to_string(index=False))
print(f"\nSaved summary table -> {summary_csv}")

 DELIVERABLE 7: CERTIFIED TEST-SET ROBUSTNESS SUMMARY
 Dataset  Test_Samples  Certified_Accuracy_Pct  Mean_Untargeted_Radius  Median_Untargeted_Radius  Min_Radius  Max_Radius  Fragile_Samples_Count  Fragile_Samples_Pct  Avg_Touched_Features
threeOf9           103                  100.00                    1.00                       1.0           1           1                    103                100.0                  1.00
   lupus            18                   66.67                    2.00                       2.0           1           4                      7                 38.9                  1.61
   mammo           193                   78.76                    1.57                       2.0           1           3                     94                 48.7                  1.57

Saved summary table -> c:\Users\rikyf\OneDrive\Desktop\rf-obdd-robustness\results\deliverable_7\deliverable_7_summary_metrics.csv
